In [ ]:
import torch
import random
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, roc_auc_score

import torch.nn.functional as F

from model import FF_TE, utils

train_set = FF_TE.FF_MNIST("train")
val_set = FF_TE.FF_MNIST("val")
test_set = FF_TE.FF_MNIST("test")
trn_loader = torch.utils.data.DataLoader(
    train_set,
    128,
    drop_last=True,
    shuffle=True,
    num_workers=4,
    persistent_workers=True,
)
val_loader = torch.utils.data.DataLoader(
    val_set,
    128,
    drop_last=True,
    shuffle=True,
    num_workers=4,
    persistent_workers=True,
)
tst_loader = torch.utils.data.DataLoader(
    test_set,
    128,
    drop_last=True,
    shuffle=True,
    num_workers=4,
    persistent_workers=True,
)

In [ ]:
def valid(net, valid_data):
    yAll = None
    outAll = None
    outProb = None

    with torch.no_grad():
        for data in valid_data:
            x, y = data
            x = x["original_sample"].cuda()
            y = y["class_labels"].cuda()
            x = x.reshape(x.shape[0], -1)

            output = net(x)
            outAll = utils.ts_append(outAll, output.argmax(1))
            outProb = utils.ts_append(outProb, output)
            yAll = utils.ts_append(yAll, y)

    acc = outAll.eq(yAll).float().mean().item()
    f1_mac = f1_score(outAll.cpu().numpy(), yAll.cpu().numpy(), average="macro")
    f1_mic = f1_score(outAll.cpu().numpy(), yAll.cpu().numpy(), average="micro")
    auc_s = roc_auc_score(yAll.cpu().numpy(), outProb.cpu().numpy(), multi_class="ovo")
    return (acc, f1_mac, f1_mic, auc_s)

In [ ]:
import torch.nn as nn


class MLPNet(nn.Module):
    def __init__(self, in_features, hidden_features, out_features):
        super(MLPNet, self).__init__()
        self.model = nn.ModuleList(
            [
                nn.Linear(in_features, hidden_features),
                nn.Linear(hidden_features, out_features),
            ]
        )
        self.relu = nn.ReLU()

    def forward(self, input):
        hidden = self.relu(self.model[0](input))
        return F.softmax(self.model[1](hidden), dim=1)

In [ ]:
from tqdm import tqdm

lr = 1e-3
wd = 1e-5
epoch = 100

model = MLPNet(784, 500, 10).cuda()
opt = torch.optim.AdamW(model.parameters(), lr=lr)

pbar = tqdm(total=epoch)
for i in range(epoch):
    pbar.set_description_str(f"Epoch: {i}/{epoch}")
    total_loss = 0
    for inputs, labels in trn_loader:
        inputs = inputs["original_sample"].cuda()
        labels = labels["class_labels"].cuda()
        opt.zero_grad()
        inputs = inputs.reshape(inputs.shape[0], -1)

        output = model(inputs)
        loss = F.nll_loss(output, labels)
        loss.backward()
        opt.step()

        total_loss += loss.item()

    trn_acc, _, _, _ = valid(model, trn_loader)
    val_acc, _, _, _ = valid(model, val_loader)
    if i % 1 == 0:
        test_acc, _, _, _ = valid(model, tst_loader)

    total_loss = total_loss / len(trn_loader)
    pbar.set_postfix(
        loss=total_loss, trn_acc=trn_acc, val_acc=val_acc, test_acc=test_acc
    )
    pbar.update(1)

with torch.no_grad():
    test_acc, test_f1_mac, test_f1_mic, test_auc = valid(model, tst_loader)
print(
    "test_acc:",
    test_acc,
    "f1 macro:",
    test_f1_mac,
    "f1 micro",
    test_f1_mic,
    "auc",
    test_auc,
)
pbar.close()